In [ ]:
import re
from itertools import count

def is_variable(x):
    return isinstance(x, str) and x.startswith("?")

def parse_atom(text):
    match = re.fullmatch(
        r"\s*(\w+)\s*\((.*?)\)\s*", text
    )

    if not match:
        raise ValueError("Invalid predicate: " + text)

    name = match.group(1)
    args = match.group(2).strip()

    return (name,) + tuple(
        x.strip() for x in args.split(",") if x.strip()
    )


def substitute(x, theta):
    if is_variable(x):
        while x in theta and theta[x] != x:
            x = theta[x]

        return x

    if isinstance(x, tuple):
        return tuple(substitute(i, theta) for i in x)

    return x


def unify(x, y, theta):
    if theta is None:
        return None

    x = substitute(x, theta)
    y = substitute(y, theta)

    if x == y:
        return theta

    if is_variable(x):
        theta[x] = y
        return theta

    if is_variable(y):
        theta[y] = x
        return theta

    if isinstance(x, tuple) and isinstance(y, tuple):

        if len(x) != len(y):
            return None

        for a, b in zip(x, y):
            theta = unify(a, b, theta)

            if theta is None:
                return None

        return theta

    return None


def standardize(rule, serial):
    mapping = {}

    def rename(x):
        if is_variable(x):
            if x not in mapping:
                mapping[x] = x + "_" + str(serial)

            return mapping[x]

        if isinstance(x, tuple):
            return tuple(rename(i) for i in x)

        return x

    premises = [rename(p) for p in rule[0]]
    conclusion = rename(rule[1])

    return premises, conclusion


def match(premises, facts, theta=None):
    if theta is None:
        theta = {}

    if not premises:
        yield theta
        return

    first = premises[0]
    remaining = premises[1:]

    for fact in facts:
        new_theta = unify(first, fact, theta.copy())

        if new_theta is not None:
            yield from match(
                remaining, facts, new_theta
            )


def fol_fc_ask(facts, rules, query):

    facts = set(facts)
    serial = count(1)

    print("\nInitial Knowledge Base:")

    for fact in sorted(facts):
        print(fact)

    print("\nGoal:", query)

    for fact in facts:
        result = unify(fact, query, {})

        if result is not None:
            print("\nGoal already proved!")
            return True

    while True:

        new = set()

        for rule in rules:

            premises, conclusion = standardize(
                rule, next(serial)
            )

            for theta in match(premises, facts):

                inferred = substitute(conclusion, theta)

                if inferred not in facts:

                    new.add(inferred)

                    print("\nRule Applied:")

                    print("Substitution:", theta)

                    print("New Fact:", inferred)

                    result = unify(inferred, query, {})

                    if result is not None:
                        print("\nGoal Successfully Proved!")
                        return True

        if not new:
            print("\nNo new facts generated.")
            print("Goal cannot be proved.")
            return False

        facts.update(new)

        print("\nUpdated Knowledge Base:")

        for fact in sorted(facts):
            print(fact)


# USER INPUT

facts = []
rules = []

print("===== FOL FORWARD CHAINING =====")

n = int(input("\nEnter number of initial facts: "))

print("\nEnter facts (Example: Fever(Patient))")

for i in range(n):
    facts.append(parse_atom(input(f"Fact {i+1}: ")))


r = int(input("\nEnter number of rules: "))

print("\nEnter rules using AND and ->")
print("Example: Fever(?x) & Sore(?x) -> Infection(?x)")

for i in range(r):

    rule = input(f"Rule {i+1}: ")

    if "->" not in rule:
        raise ValueError("Rule must contain ->")

    left, right = rule.split("->", 1)

    premises = [
        parse_atom(p)
        for p in left.split("&")
    ]

    conclusion = parse_atom(right)

    rules.append((premises, conclusion))


query = parse_atom(input("\nEnter goal/query: "))

fol_fc_ask(facts, rules, query)


===== FOL FORWARD CHAINING =====

Enter number of initial facts: 3

Enter facts (Example: Fever(Patient))
Fact 1: Fever(Patient)
Fact 2: Sore(Patient)
Fact 3:  Muscle(Patient)

Enter number of rules: 2

Enter rules using AND and ->
Example: Fever(?x) & Sore(?x) -> Infection(?x)
Rule 1: Fever(?x) & Sore(?x) -> Infection(?x)
Rule 2:  Infection(?x) & Muscle(?x) -> Tested(?x)

Enter goal/query: Tested(Patient)

Initial Knowledge Base:
('Fever', 'Patient')
('Muscle', 'Patient')
('Sore', 'Patient')

Goal: ('Tested', 'Patient')

Rule Applied:
Substitution: {'?x_1': 'Patient'}
New Fact: ('Infection', 'Patient')

Updated Knowledge Base:
('Fever', 'Patient')
('Infection', 'Patient')
('Muscle', 'Patient')
('Sore', 'Patient')

Rule Applied:
Substitution: {'?x_4': 'Patient'}
New Fact: ('Tested', 'Patient')

Goal Successfully Proved!


True